# QM640 · 07 Instrument I8: self-hosted open model (Qwen3-14B, Apache 2.0)
**Runtime → Change runtime type → L4 GPU**, then *Runtime → Run all*. About 1.5 hours (roughly 3 compute units). Uses the same prompts and test tickets as notebook 06, so I5, I6 and I8 are directly comparable. Run after notebook 06 has finished (one GPU session at a time).

I8's answers are training-eligible under the Apache 2.0 licence (decision record D-7).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import random, time, json
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
!pip -q install -U bitsandbytes
MODEL_ID = 'Qwen/Qwen3-14B'; MODEL_NAME = 'Qwen3-14B'   # open 14-billion-parameter model (Apache 2.0), self-hosted, 4-bit
BATCH = 8                        # prompts per generation batch
UNITS_PER_HOUR = 2.0             # approximate Colab compute units per hour on an L4 (check View resources)
USD_PER_UNIT = 0.10              # Colab pay-as-you-go price: about US$10 per 100 units
def load_model():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    assert torch.cuda.is_available(), 'Select an L4 GPU: Runtime > Change runtime type'
    tok = AutoTokenizer.from_pretrained(MODEL_ID); tok.padding_side = 'left'
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=q, device_map='cuda'); model.eval()
    eos = [tok.convert_tokens_to_ids('<|im_end|>'), tok.eos_token_id]
    class LM:
        def chat(self, p):   # thinking mode off: a direct answer, like the other instruments
            return tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        @torch.no_grad()
        def generate(self, prompts, max_new):
            enc = tok([self.chat(p) for p in prompts], return_tensors='pt', padding=True).to('cuda')
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                                 output_scores=True, return_dict_in_generate=True, pad_token_id=tok.pad_token_id, eos_token_id=eos)
            gen = out.sequences[:, enc['input_ids'].shape[1]:]
            lp = torch.stack(out.scores, 1).float().log_softmax(-1).gather(-1, gen.unsqueeze(-1)).squeeze(-1)
            keep = torch.cumprod((~torch.isin(gen, torch.tensor(eos + [tok.pad_token_id], device=gen.device))).int(), 1).bool()
            conf = torch.exp((lp * keep).sum(1)).tolist()                    # probability of the whole answer
            return tok.batch_decode(gen, skip_special_tokens=True), conf, enc['attention_mask'].sum(1).tolist()
    return LM()


## 1. Settings and prompts (identical to notebook 06)

In [ ]:
import json, time, gc, random, difflib, datetime, numpy as np, pandas as pd
from sklearn.metrics import f1_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
JIRA = DATA_ROOT/'processed'/'jira'; TESTS = DATA_ROOT/'processed'/'jira_tests'; RESULTS = DATA_ROOT/'results'
RAW = RESULTS/'i8_raw'; RAW.mkdir(parents=True, exist_ok=True)
SEED = 640; random.seed(SEED); FILES = sorted(JIRA.glob('*.parquet'))
LABELS = json.load(open(TESTS/'label_spaces.json'))
CLS = {'D1_issuetype': ('issuetype', 'What type of issue is this?'), 'D1_priority': ('priority', 'What priority should this issue be given?'),
       'D4_resolution': ('resolution', 'How is this issue most likely to be resolved?'), 'D5_component': ('proj_comp', 'Which project component (team) should handle this issue?')}
RET = {'D2_duplicate': 'Which of the earlier issues listed does this new issue duplicate?',
       'D6_cause': 'Which of the earlier issues listed caused the problem reported in this new issue?'}
def txt(x): return x if isinstance(x, str) else ''
def first_comp(s):
    try: c = json.loads(s)
    except Exception: return None
    return c[0] if c else None
def ticket(r, n=1200): return f"Project: {r['project']}\nSummary: {txt(r['summary'])}\nDescription: {txt(r['description'])[:n]}"
def cls_prompt(task, r, context=None):
    opts = '\n'.join(f'- {o}' for o in LABELS[task])
    ctx = ('\nSimilar earlier issues and how they were handled:\n' + context + '\n') if context else ''
    return (f"You are triaging an issue in a software issue tracker.\n\n{ticket(r)}\n{ctx}\nQuestion: {CLS[task][1]}\n"
            f"Options:\n{opts}\n\nAnswer with exactly one option from the list and nothing else.")
def ret_prompt(task, r, cands, texts):
    lines = '\n'.join(f"c{i+1}: {txt(texts.loc[c, 'summary'])} | {txt(texts.loc[c, 'description'])[:200]}" for i, c in enumerate(cands) if c in texts.index)
    return (f"New issue:\n{ticket(r)}\n\nEarlier issues:\n{lines}\nnone: None of the listed issues\n\nQuestion: {RET[task]}\n"
            "Answer with the identifier only (for example c3, or none).")
def to_label(text, space):
    t = text.strip().strip('.').strip('"').strip("'").splitlines()[0].strip() if text.strip() else ''
    low = {o.lower(): o for o in space}
    if t.lower() in low: return low[t.lower()]
    m = difflib.get_close_matches(t, space, n=1, cutoff=0.6)
    return m[0] if m else 'Other'
def wilson(k, n): lo, hi = proportion_confint(k, n, method='wilson'); return f'{100*lo:.1f}–{100*hi:.1f}'
def coverage(conf, ok, p):
    o = np.argsort(-conf); prec = np.cumsum(ok[o])/np.arange(1, len(o)+1); idx = np.where(prec >= p)[0]
    return round(100*(idx.max()+1)/len(o), 1) if len(idx) else 0.0
def ece(conf, ok, m=10):
    e = 0
    for a, b in zip(np.linspace(0, 1, m+1)[:-1], np.linspace(0, 1, m+1)[1:]):
        s = (conf > a) & (conf <= b)
        if s.any(): e += s.mean()*abs(ok[s].mean() - conf[s].mean())
    return round(e, 3)
tick = pd.read_parquet(TESTS/'jira_test_tickets.parquet').drop_duplicates('key').set_index('key', drop=False)
tests = {t: pd.read_csv(TESTS/f'testset_jira_{t}.csv') for t in list(CLS) + list(RET)}
HAS_CONTEXT = (TESTS/'context_i2_i3.csv').exists()
print(len(tick), 'test tickets; context from notebook 05 available:', HAS_CONTEXT)

## 2. Load Qwen3-14B in 4-bit
Thinking mode is switched off, so the model answers directly like the other instruments.

In [ ]:
t_gpu0 = time.time()
lm = load_model()

## 3. Run helper (resumable; records GPU time per decision)

In [ ]:
def run(name, items, max_new_tokens):
    """items: list of (key, prompt). Saves one JSON line per ticket; reruns skip finished tickets."""
    f = RAW/f'{name}.jsonl'; done = {}
    if f.exists():
        for l in open(f): o = json.loads(l); done[o['key']] = o
    todo = [(k, p) for k, p in items if k not in done]
    t0 = time.time()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b+BATCH]
        outs, confs, ntok = lm.generate([p for _, p in chunk], max_new_tokens)
        with open(f, 'a') as fh:
            for (k, _), o, c, n in zip(chunk, outs, confs, ntok):
                rec = {'key': k, 'text': o, 'conf': c, 'prompt_tokens': n}; done[k] = rec; fh.write(json.dumps(rec) + '\n')
    secs = time.time() - t0
    print(f'{name}: {len(todo)} new, {len(done)} total, {secs:.0f}s')
    return done, secs
timing = {}

## 4. I8 on the classification test sets

In [ ]:
for t in CLS:
    items = [(k, cls_prompt(t, tick.loc[k])) for k in tests[t]['key']]
    _, timing[('I8', t)] = run(f'i8_{t}', items, 24)

## 5. I8 on duplicate (D2) and cause (D6) search, picking from the I1 shortlist

In [ ]:
need = set()
for t in RET: need |= {c for s in pd.read_csv(TESTS/f'i1_predictions_{t}.csv')['i1_top10'] for c in json.loads(s)}
texts = pd.concat([tick[['key', 'summary', 'description']]] + [pd.read_parquet(f, columns=['key', 'summary', 'description'], filters=[('key', 'in', list(need - set(tick.index)))]) for f in FILES]).drop_duplicates('key').set_index('key')
shortlist = {t: dict(zip(*pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_top10']].values.T)) for t in RET}
for t in RET:
    items = [(k, ret_prompt(t, tick.loc[k], json.loads(shortlist[t][k]), texts)) for k in tests[t]['key'] if k in shortlist[t]]
    _, timing[('I8', t)] = run(f'i8_{t}', items, 6)

## 6. RQ3: with company history (D5 with similar earlier issues; D2/D6 on the hybrid shortlist)

In [ ]:
if HAS_CONTEXT:
    ctx = pd.read_csv(TESTS/'context_i2_i3.csv')
    ctick = pd.read_parquet(TESTS/'context_tickets.parquet').drop_duplicates('key').set_index('key')
    ctick['proj_comp'] = [f'{p} | {c}' if c else None for p, c in zip(ctick['project'], ctick['components'].map(first_comp))]
    NICE = {'issuetype': 'type', 'priority': 'priority', 'resolution': 'resolution', 'proj_comp': 'component'}
    def context_text(t, k, n=5):
        row = ctx[(ctx['decision'] == t) & (ctx['key'] == k)]
        if row.empty: return None
        out = []
        for c in json.loads(row['i2_top10'].iloc[0])[:n]:
            if c in ctick.index:
                r = ctick.loc[c]; out.append(f"- {txt(r['summary'])[:150]} -> {NICE[CLS[t][0]]}: {r[CLS[t][0]]}")
        return '\n'.join(out)
    t = 'D5_component'
    items = [(k, cls_prompt(t, tick.loc[k], context_text(t, k))) for k in tests[t]['key']]
    _, timing[('I8+ctx', t)] = run(f'i8_ctx_{t}', items, 24)
    hyb = {t: dict(zip(ctx[ctx['decision'] == t]['key'], ctx[ctx['decision'] == t]['hybrid_top10'])) for t in RET}
    extra = {c for t in RET for s in hyb[t].values() if isinstance(s, str) for c in json.loads(s)} - set(texts.index)
    texts = pd.concat([texts.reset_index()] + [pd.read_parquet(f, columns=['key', 'summary', 'description'], filters=[('key', 'in', list(extra))]) for f in FILES]).drop_duplicates('key').set_index('key')
    for t in RET:
        items = [(k, ret_prompt(t, tick.loc[k], json.loads(hyb[t][k]), texts)) for k in tests[t]['key'] if isinstance(hyb[t].get(k), str)]
        _, timing[('I8+ctx', t)] = run(f'i8_ctx_{t}', items, 6)
gpu_seconds = time.time() - t_gpu0

## 7. Results: classification

In [ ]:
def load(name):
    f = RAW/f'{name}.jsonl'
    return {json.loads(l)['key']: json.loads(l) for l in open(f)} if f.exists() else {}
i1 = {t: dict(zip(*pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_pred']].values.T)) for t in CLS}
rows, preds = [], []
runs = [('I8', f'i8_{t}', t) for t in CLS] + [('I8 + I2 context', 'i8_ctx_D5_component', 'D5_component')]
for inst, name, t in runs:
    a = load(name)
    if not a: continue
    ts = tests[t][tests[t]['key'].isin(a)]; y = ts['label'].values
    p = np.array([to_label(a[k]['text'], LABELS[t]) for k in ts['key']]); conf = np.array([a[k]['conf'] for k in ts['key']])
    ok = p == y; ok1 = np.array([i1[t].get(k) == lab for k, lab in zip(ts['key'], y)]); b, c = int((ok1 & ~ok).sum()), int((~ok1 & ok).sum())
    sec = timing.get((inst.split(' ')[0] + ('+ctx' if 'context' in inst else ''), t), np.nan)
    rows.append({'Decision': t, 'Instrument': f'{inst} ({MODEL_NAME})', 'Test rows': len(y), 'Accuracy %': round(100*ok.mean(), 1), '95% CI (Wilson)': wilson(int(ok.sum()), len(y)),
                 'Macro-F1': round(f1_score(y, p, average='macro', zero_division=0), 3), 'I1 accuracy %': round(100*ok1.mean(), 1),
                 'McNemar p (vs I1)': round(mcnemar([[0, b], [c, 0]], exact=True).pvalue, 4), 'Coverage % at 98% precision': coverage(conf, ok, .98),
                 'Coverage % at 90% precision': coverage(conf, ok, .90), 'ECE': ece(conf, ok), 'Unmatched answers %': round(100*np.mean([to_label(a[k]['text'], LABELS[t]) == 'Other' and a[k]['text'].strip().lower() != 'other' for k in ts['key']]), 1),
                 'GPU seconds per ticket': round(sec/len(y), 3) if sec == sec else None,
                 'Cost US$ per 1,000 tickets': round(1000*sec/len(y)/3600*UNITS_PER_HOUR*USD_PER_UNIT, 3) if sec == sec else None})
    preds.append(pd.DataFrame({'instrument': inst, 'decision': t, 'key': ts['key'], 'label': y, 'pred': p, 'conf': conf}))
slm = pd.DataFrame(rows); slm.to_csv(RESULTS/'rq1_jira_i8.csv', index=False)
pd.concat(preds).to_csv(TESTS/'i8_predictions_classification.csv', index=False); slm

## 8. Results: duplicate and cause search

In [ ]:
rrows = []
for inst, pre in (('I8 re-rank of I1 shortlist', 'i8_'), ('I8 re-rank of I1+I2 hybrid shortlist', 'i8_ctx_')):
    for t in RET:
        a = load(f'{pre}{t}')
        if not a: continue
        sl = shortlist[t] if pre == 'i8_' else hyb[t]
        hit = n = 0
        for k, ans in zip(tests[t]['key'], tests[t]['answers']):
            if k not in a or not isinstance(sl.get(k), str): continue
            n += 1; cands = json.loads(sl[k]); out = a[k]['text'].strip().lower()
            idx = int(''.join(ch for ch in out.split()[0] if ch.isdigit()) or 0) if out.startswith('c') else 0
            hit += bool(0 < idx <= len(cands) and cands[idx-1] in set(json.loads(ans)))
        rrows.append({'Decision': t, 'Instrument': f'{inst} ({MODEL_NAME})', 'Test rows': n, 'Recall@1 %': round(100*hit/n, 1), '95% CI': wilson(hit, n)})
rret = pd.DataFrame(rrows); rret.to_csv(RESULTS/'rq1_jira_i8_rerank.csv', index=False); rret

## 9. Cost and summary

In [ ]:
cost = pd.DataFrame([{'Item': 'All I8 inference this session', 'GPU minutes': round(gpu_seconds/60, 1),
                      'Approx. compute units': round(gpu_seconds/3600*UNITS_PER_HOUR, 2),
                      'Approx. US$': round(gpu_seconds/3600*UNITS_PER_HOUR*USD_PER_UNIT, 2)}])
cost.to_csv(RESULTS/'rq1_i8_cost.csv', index=False)
def mdt(d):
    try: return d.to_markdown(index=False)
    except Exception: return d.to_string()
lines = [f'# Step 07 summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', f'Model: {MODEL_NAME} (4-bit)', '',
         '## Classification', mdt(slm), '', '## Duplicate and cause re-ranking', mdt(rret), '', '## Cost', mdt(cost)]
(RESULTS/'step07_summary.md').write_text('\n'.join(lines)); print('\n'.join(lines))